In [0]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F

def run_validation(df, rules, table_name):
    run_id = str(uuid.uuid4())

    df = df.withColumn("_raw_payload", F.to_json(F.struct(*df.columns)))

    rule_cols = []
    validated_df = df
    for rule in rules:
        col_name = f"_rule_{rule['name']}"
        validated_df = validated_df.withColumn(col_name, rule["condition"])
        rule_cols.append(col_name)

    validated_df = validated_df.withColumn(
        "_is_valid",
        F.array_min(F.array(*[F.col(c).cast("int") for c in rule_cols])) == 1
    )

    # Single-pass aggregation instead of 9+ separate .count() actions
    agg_exprs = [F.count(F.lit(1)).alias("total_records")]
    agg_exprs += [F.sum((~F.col(c)).cast("int")).alias(c) for c in rule_cols]
    agg_exprs += [F.sum(F.col("_is_valid").cast("int")).alias("_valid_count")]
    stats = validated_df.agg(*agg_exprs).collect()[0]

    total_records = stats["total_records"]
    valid_count = stats["_valid_count"]
    invalid_count = total_records - valid_count

    dq_results = []
    for rule in rules:
        col_name = f"_rule_{rule['name']}"
        failed = stats[col_name]
        failure_pct = round((failed / total_records) * 100, 4) if total_records > 0 else 0.0
        dq_results.append({
            "run_id": run_id,
            "table_name": table_name,
            "rule_name": rule["name"],
            "records_checked": total_records,
            "records_failed": failed,
            "failure_percentage": failure_pct,
            "execution_timestamp": datetime.now()
        })
    dq_results_df = df.sparkSession.createDataFrame(dq_results)

    valid_df = validated_df.filter(F.col("_is_valid")).drop(*rule_cols, "_is_valid", "_raw_payload")
    invalid_df = validated_df.filter(~F.col("_is_valid"))

    failed_rule_names = [F.when(~F.col(f"_rule_{r['name']}"), F.lit(r["name"])) for r in rules]
    quarantine_df = (
        invalid_df
        .withColumn("failed_rule_list", F.array_except(F.array(*failed_rule_names), F.array(F.lit(None).cast("string"))))
        .withColumn("failed_rule", F.array_join("failed_rule_list", ","))
        .withColumn("failure_reason", F.concat(F.lit("Failed rules: "), F.col("failed_rule")))
        .withColumn("raw_payload", F.col("_raw_payload"))
        .withColumn("source_file", F.col("_source_file"))
        .withColumn("detected_at", F.current_timestamp())
        .select("event_id", "trip_id", "failed_rule", "failure_reason", "raw_payload", "source_file", "detected_at")
    )

    return valid_df, quarantine_df, dq_results_df, validated_df, valid_count, invalid_count